# Adjoint capture analysis: a stream, a lake, and the sea

A well pumping near the coast competes for water with every other outlet the
aquifer has. Some of what it produces is released from aquifer storage, and the
rest is **capture** — water that would otherwise have discharged to a stream, a
lake, or the sea, or that those features now lose to the aquifer. This notebook
splits the capture of one well three ways, using **adjoint-state sensitivity
analysis** with [mf6adj](https://github.com/INTERA-Inc/mf6adj) on the coastal
synthetic valley.

By the end of this notebook you will be able to:

- represent a saltwater coast in a constant-density model with an **equivalent
  freshwater head**,
- write one mf6adj **performance measure** per competing outlet and solve them
  all from a single backward solve,
- read the capture fraction each outlet supplies, and map it for a well placed
  anywhere in the aquifer,
- find where in the valley the sea takes over from the stream as the main source
  of a well's water,
- measure how much more seawater the coast delivers for a foot of sea-level
  rise, and
- check every measure with a small-perturbation two-run difference and account
  for the whole of the pumping in the water budget.

## The three outlets and one backward solve

The direct way to find out how sensitive a model result is to a parameter is to
change the parameter and run the model again. That costs one run per parameter,
so a sensitivity map over a 5-layer, 40-by-25 grid would cost thousands of runs.

The adjoint approach turns the problem around. Name the single model output you
care about — the **performance measure** — and solve the flow equations backward
in time once. That one backward solve returns the sensitivity of that measure to
every parameter in every cell: hydraulic conductivity, storage, recharge,
boundary conductance, and well rates. The cost is one backward solve per
measure, rather than one forward run per parameter, so three outlets cost three
backward solves and one forward run, whatever the size of the grid.

The sensitivity of a boundary flow to a well rate is the **capture fraction**
directly: it is the change in that boundary's flow per unit of pumping, so a
value of 1 means every unit pumped is taken from that feature and 0 means none of
it is.

mf6adj drives MODFLOW 6 through its Application Programming Interface (**API**)
to collect the matrices it needs, so the model itself is unmodified. The method
and its verification are described in Hayek and others (2025), *MF6-ADJ: A
Non-Intrusive Adjoint Sensitivity Capability for MODFLOW 6*, Groundwater 63(6),
874–888. [mf6-adj-capture](mf6-adj-capture.ipynb) works through the same method
on the inland valley, with two outlets instead of three.

Import the packages this notebook uses. `mf6adj` provides the adjoint solver and
`mf6_adj_helpers` collects the workspace setup and the sensitivity readers shared
by the adjoint notebooks. `%matplotlib inline` has to come first, or only the
first figure drawn inside a `flopy.plot.styles` context is rendered.

In [ ]:
%matplotlib inline

import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries

Locate the MODFLOW 6 shared library and executable in the active environment.
mf6adj drives the model through the shared library (`libmf6`), so both paths are
needed: the executable for the ordinary forward runs and the library for the
adjoint.

In [ ]:
lib_name, mf6_exe = find_mf6_libraries()
print(f"library:    {lib_name.name}")
print(f"executable: {mf6_exe.name}")

## Prepare the coastal model

Use the **coastal advanced** synthetic-valley model at an annual sampling
frequency. It represents the valley with the advanced hydrologic packages:
streamflow routing (**SFR**) for the river, a lake (**LAK**), unsaturated-zone
flow (**UZF**) for recharge and evapotranspiration, and the water mover
(**MVR**) to route water between them. A general-head boundary (**GHB**) along
the southern row is the sea, built in
[mf6-coastal-ghb](mf6-coastal-ghb.ipynb). The 21 stress periods start with a
long steady spin-up followed by 20 annual periods, and the model is in feet and
days.

The model needs no preparation for the adjoint. The production wells are
multi-aquifer wells (**MAW**), whose heads MODFLOW 6 solves along with the
aquifer heads, so each well is already one of the rows of the matrix mf6adj
differentiates. SFR, LAK, and UZF are solved in the outer (Picard) iteration
instead, so mf6adj borders the adjoint system with their own equations, which is
where the warnings further down come from.

`prepare_model()` copies the shipped model into `models/`, sets the prediction
well to a rate of zero, and raises the coastal boundary heads to equivalent
freshwater heads.

In [ ]:
ws = adjh.prepare_model(
    "adj-coastal-capture",
    variant="coastal-advanced",
    prediction_rate=0.0,
    equivalent_freshwater=True,
)
adjh.run_model(ws, mf6_exe)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(ws), verbosity_level=0)
gwf = sim.get_model()
nper = sim.tdis.nper.data
nlay, nrow, ncol = gwf.dis.nlay.data, gwf.dis.nrow.data, gwf.dis.ncol.data
print(f"grid:           {nlay} layers, {nrow} rows, {ncol} columns")
print(f"stress periods: {nper}")
print(f"packages:       {', '.join(sorted(gwf.package_names))}")

This first run is the **baseline**: the prediction well is present but pumps at a
rate of zero. The sensitivities describe how the model responds to a small change
away from this baseline, which makes the capture fraction a property of the
aquifer and the well's position rather than of one particular pumping rate.

### Why the boundary head is not zero

Seawater is denser than fresh groundwater, so a column of it presses down harder
than the same column of fresh water would. This model carries a single constant
fluid density, and the way to represent the weight of the sea in such a model is
to raise the boundary head until a column of fresh water would press down just as
hard. That raised head is the **equivalent freshwater head**, and it grows with
depth below sea level:

```
h_f = h_s + (rho_s / rho_f - 1) * (h_s - z)
```

where `h_s` is sea level, `z` is the elevation of the boundary, and the density
ratio `rho_s / rho_f` is 1,024.5 over 1,000 for seawater at 35 kilograms per
cubic meter. Print what that gives in each layer. Leaving the boundary at sea
level everywhere would understate the resistance the sea offers to outflow, and
the understatement grows with depth.

In [ ]:
ghb_spd = gwf.get_package("ghb-1").stress_period_data.get_data(0)
# (layer, row, column), boundary head, conductance, elevation, concentration
print(f"first boundary: {tuple(ghb_spd[0])}")
print()
print("layer   cells   elevation, ft     boundary head, ft   conductance, ft2/d")
for k in range(nlay):
    sel = ghb_spd[[cellid[0] == k for cellid in ghb_spd["cellid"]]]
    print(
        f"  {k + 1}      {len(sel):3d}   "
        f"{sel['elevation'].min():8.1f} to {sel['elevation'].max():7.1f}"
        f"    {sel['bhead'].min():6.3f} to {sel['bhead'].max():6.3f}"
        f"     {sel['cond'].sum():12,.0f}"
    )
print(f"\ntotal coastal conductance: {ghb_spd['cond'].sum():,.0f} ft2/d")

**What to look for.** The boundary head rises from 0.061 ft at the 2.5-ft
elevation of the layer-1 boundaries to between 3.1 and 5.8 ft in layer 5, where
the boundaries sit 126 to 236 ft below sea level. Every one of those heads is
above sea level, so the sea resists outflow more than a zero head would and can
drive water inland where the aquifer head is low enough. Layers 4 and 5 carry
1,859,000 of the 2,227,000 ft2/d of coastal conductance, or 84 percent of it,
because the two deep layers are both thick and permeable.

### Look at the model

Map the features that compete for the well's water. Plot the lake cells and the
stream reaches with `.plot_bc()`, mark the coastal boundary along the southern
row, and mark the two production wells and the prediction well. Cell IDs are
`(layer, row, column)` counting from zero, as everywhere in FloPy.

In [ ]:
pred_cell = (4, 34, 15)  # zero-based (layer, row, column)
prod_cells = adjh.package_cells(gwf, "pwell")
sfr_cells = adjh.package_cells(gwf, "sfr-1")
lak_cells = adjh.package_cells(gwf, "lak-1")
coast_cells = adjh.package_cells(gwf, "ghb-1")

xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
px, py = xc[pred_cell[1], pred_cell[2]], yc[pred_cell[1], pred_cell[2]]
print("distance from the prediction well to the nearest")
for label, cells in (
    ("coast", coast_cells),
    ("stream reach", sfr_cells),
    ("lake cell", lak_cells),
):
    d = min(np.hypot(xc[i, j] - px, yc[i, j] - py) for _, i, j in cells)
    print(f"  {label:13s} {d:7,.0f} ft ({d / 5280.0:.2f} mi)")

with flopy.plot.styles.USGSMap():
    fig, ax = plt.subplots(figsize=(4.5, 7), layout="constrained")
    mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
    mm.plot_grid(lw=0.2, color="0.85")
    mm.plot_bc("LAK", color="tab:blue")
    mm.plot_bc("SFR", color="tab:cyan")
    mm.plot_ibound()
    coast_rc = sorted({(i, j) for _, i, j in coast_cells})
    ax.plot(
        [xc[i, j] for i, j in coast_rc],
        [yc[i, j] for i, j in coast_rc],
        "s",
        color="tab:green",
        ms=6,
    )
    for _, i, j in prod_cells:
        ax.plot(xc[i, j], yc[i, j], "ko", ms=7)
    ax.plot(px, py, "r*", ms=16, label="prediction well")
    ax.plot([], [], "ko", ms=7, label="production wells")
    ax.plot([], [], "s", color="tab:blue", label="lake")
    ax.plot([], [], "s", color="tab:cyan", label="stream")
    ax.plot([], [], "s", color="tab:green", label="coast")
    ax.legend(loc="upper right", fontsize=8)
    ax.set_xlabel("x, in feet")
    ax.set_ylabel("y, in feet")
    ax.set_title("Coastal synthetic valley: the outlets and the wells")

**What to look for.** The lake occupies the northern third of the valley, the
stream runs the length of it, and the coast is the southern row, broken only at
the column the stream leaves through. The prediction well (red star) is 2,500 ft
from the coast, 3,500 ft from the nearest stream reach, and 9,657 ft from the
nearest lake cell. The sea is the closest outlet to this well and the lake the
farthest.

## Define one performance measure per outlet

An mf6adj performance measure lists the model outputs to combine. mf6adj reads
it from a plain-text block with one line per entry:

```
# kper kstp layer row column  package  form  weight  observed
  21    1     5    40    12   ghb-1   direct  1.0    -1.0e+30
```

The `package` field is either `head` — for the head in that cell — or the name of
a boundary package from the model name file, which selects the flow between that
package and the cell. `direct` means the measure is the value itself rather than a
residual against an observation, so the `observed` field is unused.

Give `mf6adj.write_performance_measures()` each measure as a dictionary of
columns: the cells in `cellid`, exactly as FloPy returns them, and the time in
`kper` and `kstp`. All of these are zero-based, and the writer converts them to
the one-based indices of the file. The file is written as ascii text unless you
pass `format="hdf5"`.

Build three measures at the last stress period, each summing the exchange
between one package and the aquifer over all of that package's cells. Together
they are the whole of the surface water and seawater that pumping can capture.

In [ ]:
last = nper - 1  # zero-based
measures = {}
for name, package, cells in (
    ("swgw", "sfr-1", sfr_cells),
    ("lakegw", "lak-1", lak_cells),
    ("coastgw", "ghb-1", coast_cells),
):
    print(f"{name:8s} {package:6s} {len(cells):4d} cells")
    measures[name] = {
        "cellid": cells,
        "kper": last,
        "kstp": 0,  # every period is a single time step
        "pm_type": package,
    }
adj_file = mf6adj.write_performance_measures(ws / "capture.adj", measures)
print()
print(adj_file.read_text()[:200] + "...")

## Solve the forward and adjoint problems

Create the `mf6adj.Mf6Adj` object with the measure file and the shared library.
`solve_forward_model()` runs MODFLOW 6 through the API and saves the solution
matrix and boundary terms at every time step. `solve_adjoint()` then sweeps
backward through those saved time steps, once per measure, and all three measures
come out of that single forward run. Always call `finalize()` to release the
library. `logging_level="WARNING"` keeps the per-time-step progress messages out
of the notebook.

Keep the level at `WARNING` rather than raising it, because that is where mf6adj
reports that part of a package's derivative is not formed. This model draws
three such warnings.

In [ ]:
adj = mf6adj.Mf6Adj(
    adj_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(ws),
)
adj.solve_forward_model()
sensitivities = adj.solve_adjoint()
adj.finalize()

print(f"measures solved: {', '.join(sensitivities)}")
print(f"parameters:      {', '.join(sensitivities['coastgw'].columns)}")

**What to look for.** The first warning names two packages mf6adj forms no terms
for at all, UZF and the water mover. Their exchange with the aquifer is carried
by the flow matrix, so the head sensitivities account for it, but neither package
can be measured and neither one's response to pumping enters a derivative. The
other two warnings name the parts of the lake derivative that are missing: the
inflow the mover routes into the lake, which follows the state of the package
supplying it, and the lake's horizontal connections, whose conductance and wetted
area follow the stage through the saturated fraction of the cell. Neither the
stream nor the coast draws a warning, so the coast measure is the one boundary
here whose derivative is formed in full.

Each measure returns a table with one row per model cell and one column per
parameter — hydraulic conductivity (`k11`, `k33`), the two storage properties
(specific storage `ss` and specific yield `sy`), recharge, the stage and
conductance of the stream and the lake, the boundary head and conductance of the
coast (`ghb-1_bhead`, `ghb-1_cond`), and the well rate (`wel6_q`). Every one of
those columns came from the one backward solve.

## Read the capture fractions

Read the `wel6_q` sensitivity at the prediction well cell for each measure. Sum
the per-period sensitivities over the periods the well actually pumps, 12 through
21, because each period's value is that period's contribution to the measure at
the final period. Print the baseline flow through each outlet alongside, so the
fractions can be read against what each feature carries.

In [ ]:
active = range(11, nper)  # zero-based periods the prediction well pumps
capture = {}
for measure in measures:
    capture[measure] = adjh.total_sensitivity(
        ws, measure, "wel6_q", cell=pred_cell, periods=active
    )

labels = {"swgw": "stream", "lakegw": "lake", "coastgw": "coast"}
terms = {"swgw": "SFR(SFR-1)", "lakegw": "LAK(LAK-1)", "coastgw": "GHB(GHB-1)"}
print("outlet   capture fraction   baseline net flow, ft3/d")
for measure, label in labels.items():
    net = adjh.budget_net(ws, terms[measure])[last]
    print(f"{label:8s} {capture[measure]:12.4f}       {net:16,.0f}")
print(f"{'total':8s} {sum(capture.values()):12.4f}")

**What to look for.** The well takes 0.566 of its water from the sea, 0.345 from
the stream, and 0.0006 from the lake, which is under 0.1 percent of the pumping.
The three together are 0.912, so only 0.088 of the pumping comes from anywhere
else. The signs are negative because the well rate is itself negative: pumping
harder makes each feature give up more water.

The baseline flows show that neither the size nor the direction of a flow says
much about what a new well will draw on. The aquifer discharges 1,207,000 ft3/d
to the stream, and the sea is a net source of 222,000 ft3/d to the aquifer rather
than an outlet at all, and the sea still supplies most of this well's water.
Distance decides it: the coast is 2,500 ft away, the nearest stream reach 3,500
ft, and the lake 9,657 ft, which is far enough to be out of reach.

### Map capture for a well anywhere in the aquifer

The `wel6_q` sensitivity is not only defined at the prediction well. mf6adj
returns it in every cell, so the same three backward solves say what fraction of
a well's pumping each outlet would supply wherever the well were placed. Map the
negative of the composite `wel6_q` sensitivity for each measure, which is the
capture fraction for a well pumping over every stress period, in the middle
layer.

Capture can be negative, which is a feature giving up less water to the aquifer
under pumping than it did before. The color scale runs viridis from 0 to the
maximum and red-orange below zero. The stream and coast maps share a scale of 0
to 1, and the lake map is scaled to its own largest value, which is about sixty
times smaller.

In [ ]:
# viridis above zero and inferno below it, both dark at zero and brighter with
# size, so 0 to 1 reads as ordinary viridis and negative values as red-orange
signed = mpl.colors.ListedColormap(
    np.vstack(
        (
            plt.cm.inferno(np.linspace(0.85, 0.15, 128)),
            plt.cm.viridis(np.linspace(0.0, 1.0, 128)),
        )
    )
)

layer = 2  # zero-based
capture_maps = {
    measure: -adjh.composite_sensitivity(ws, measure, "wel6_q") for measure in measures
}
panels = {"swgw": "a", "lakegw": "b", "coastgw": "c"}

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic(
        [list(panels.values())], figsize=(11, 6), layout="constrained"
    )
    for measure, panel in panels.items():
        capture_map = capture_maps[measure]
        vmax = 1.0 if measure != "lakegw" else np.abs(capture_map[layer]).max()
        ax = axd[panel]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=layer)
        cb = mm.plot_array(capture_map[layer], cmap=signed, vmin=-vmax, vmax=vmax)
        mm.plot_ibound()

        # the stream, lake and coast are in layer 1, so outline their cells
        # rather than plotting them with plot_bc
        for cells, color in (
            (sfr_cells, "tab:cyan"),
            (lak_cells, "white"),
            (coast_cells, "tab:green"),
        ):
            rc = sorted({(i, j) for _, i, j in cells})
            ax.plot(
                [xc[i, j] for i, j in rc],
                [yc[i, j] for i, j in rc],
                "s",
                mfc="none",
                mec=color,
                ms=4,
            )
        ax.plot(px, py, "r*", ms=14)
        ax.set_xlabel("x, in feet")
        flopy.plot.styles.heading(ax=ax, letter=panel, heading=labels[measure])
        fig.colorbar(cb, ax=ax, shrink=0.4, label="capture fraction, dimensionless")
    axd["a"].set_ylabel("y, in feet")

active_cells = gwf.dis.idomain.array[layer] > 0
top_cells = gwf.dis.idomain.array[0] > 0
for measure, label in labels.items():
    layer_map = capture_maps[measure][layer]
    difference = np.abs(capture_maps[measure][0] - layer_map)[top_cells]
    print(
        f"{label:8s} layer {layer + 1}: "
        f"{layer_map[active_cells].min():7.4f} to {layer_map[active_cells].max():7.4f}"
        f"   layers 1 and {layer + 1} differ by "
        f"{np.percentile(difference, 95):.4f} at 95% of cells, "
        f"at most {difference.max():.4f}"
    )

**What to look for.** Panel c is brightest along the southern row, where a well
takes up to 0.973 of its water from the sea, and the brightness fades north.
Panel a is brightest in a band along the stream, up to 0.930, and fades south
toward the coast and north toward the lake. The two are complementary: every
well is somewhere on a gradient between them.

Panel b is 0.010 or less everywhere, and negative over the northern third of the
valley, most strongly beneath and west of the lake, reaching -0.017. The lake has
no outlet and receives runoff from UZF through the mover, and that runoff falls
when pumping lowers the water table. Less runoff lowers the lake stage and the
lake leaks less, so a well near the lake takes less water from it than it did
before pumping.

The depth of the well hardly matters. Layers 1 and 3 differ by 0.036 or less at
95 percent of the cells, and by at most 0.054, for all three measures, so how
deep the well is set changes its capture far less than where it is drilled.

### Where the sea takes over from the stream

Read the two capture maps down the column the prediction well sits in, from the
northern end of the valley to the coast, to find where the sea overtakes the
stream. Plot both fractions and their sum with distance from the coast, and count
how much of the valley each outlet dominates.

In [ ]:
col = pred_cell[2]
rows = [i for i in range(nrow) if active_cells[i, col]]
distance = np.array([yc[i, col] - yc[nrow - 1, col] for i in rows])
profile = {
    measure: np.array([capture_maps[measure][layer, i, col] for i in rows])
    for measure in measures
}
total = sum(profile.values())

# the row where the two curves cross, read from the change of sign
gap = profile["coastgw"] - profile["swgw"]
crossing = np.flatnonzero(np.diff(np.sign(gap)) != 0)[0]
near, far = sorted(distance[crossing : crossing + 2])
print(
    f"the sea overtakes the stream between rows {rows[crossing] + 1} and "
    f"{rows[crossing + 1] + 1}, {near:,.0f} to {far:,.0f} ft inland"
)
print(
    f"all three outlets together supply {total.min():.3f} to {total.max():.3f} "
    f"down this column, and {total[rows.index(pred_cell[1])]:.3f} at the well"
)

winner = np.argmax(np.stack([capture_maps[m][layer] for m in measures]), axis=0)
for index, measure in enumerate(measures):
    share = (winner == index)[active_cells].mean()
    print(f"{labels[measure]:8s} supplies the most water at {share:5.1%} of the valley")

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
    for measure, color in (
        ("coastgw", "tab:green"),
        ("swgw", "tab:cyan"),
        ("lakegw", "tab:blue"),
    ):
        ax.plot(distance, profile[measure], color=color, lw=2, label=labels[measure])
    ax.plot(distance, total, "k--", lw=1.2, label="all three")
    well_distance = py - yc[nrow - 1, col]
    ax.axvline(well_distance, color="0.4", lw=0.8)
    ax.text(
        well_distance,
        0.02,
        " prediction well",
        fontsize=8,
        rotation=90,
        va="bottom",
    )
    ax.set_xlabel("distance inland from the coast, in feet")
    ax.set_ylabel("capture fraction, dimensionless")
    ax.set_xlim(0, distance.max())
    ax.set_ylim(0, 1.0)
    ax.legend(loc="upper center", ncols=4, fontsize=8)
    ax.set_title(f"Capture down column {col + 1}, layer {layer + 1}")

**What to look for.** The sea overtakes the stream between rows 32 and 33, 3,500
to 4,000 ft inland, and the prediction well sits 2,500 ft inland, on the seaward
side of that divide. Over the whole layer the stream supplies the most water at
79.9 percent of the valley and the sea at 20.1 percent, and the lake supplies the
most nowhere.

The dashed curve is what all three outlets supply together. It reaches 0.992 at
the coast, falls to 0.100 at the northern end of the column, and is 0.912 at the
well. Where the curve is low the pumping is supplied
instead by aquifer storage and by infiltration that UZF would otherwise have
rejected as runoff, and both of those matter most in the middle of the valley,
farthest from every outlet.

## What a foot of sea-level rise does

The same backward solves returned the sensitivity of each measure to the coastal
boundary itself: `ghb-1_bhead` is the derivative with respect to the boundary
head, and `ghb-1_cond` the derivative with respect to its conductance. A rise in
sea level is a rise in that boundary head, so `ghb-1_bhead` answers a question no
capture fraction can: how much more seawater the coast delivers, and how much
more water the stream receives, when the sea comes up.

Sum the boundary-head sensitivity by layer to see which part of the coast carries
the response, and weight the conductance sensitivity by the conductance itself to
find what a 1 percent error in the conductance would cost.

In [ ]:
cond = np.zeros((nlay, nrow, ncol))
for record in ghb_spd:
    cond[tuple(record["cellid"])] = record["cond"]

bhead_sens = {
    measure: adjh.composite_sensitivity(ws, measure, "ghb-1_bhead")
    for measure in measures
}
print("outlet   per ft of boundary head   per 1% more coastal conductance")
for measure, label in labels.items():
    dcond = adjh.composite_sensitivity(ws, measure, "ghb-1_cond")
    print(
        f"{label:8s} {np.nansum(bhead_sens[measure]):16,.0f} ft3/d"
        f" {0.01 * np.nansum(cond * dcond):20,.0f} ft3/d"
    )
deep = np.nansum(bhead_sens["coastgw"][3:]) / np.nansum(bhead_sens["coastgw"])
print(
    f"\nlayers 4 and 5 hold {cond[3:].sum() / cond.sum():.1%} of the coastal "
    f"conductance and carry {deep:.1%} of the coastal response"
)

with flopy.plot.styles.USGSPlot():
    fig, ax = plt.subplots(figsize=(7, 4.5), layout="constrained")
    width = 0.38
    offsets = {"coastgw": -0.5 * width, "swgw": 0.5 * width}
    for measure, color in (("coastgw", "tab:green"), ("swgw", "tab:cyan")):
        by_layer = [np.nansum(bhead_sens[measure][k]) for k in range(nlay)]
        ax.bar(
            np.arange(1, nlay + 1) + offsets[measure],
            by_layer,
            width,
            color=color,
            edgecolor="k",
            lw=0.4,
            label=labels[measure],
        )
    ax.axhline(0.0, color="k", lw=0.8)
    ax.set_xlabel("model layer")
    ax.set_ylabel("change in flow per foot of boundary head, in ft$^3$/d")
    ax.set_xticks(range(1, nlay + 1))
    ax.legend(loc="lower left", fontsize=8)
    ax.set_title("Response of each outlet to a higher sea")

**What to look for.** A foot of boundary head brings in 175,000 ft3/d more water
across the coast and sends 149,000 ft3/d more into the stream, which is 12
percent of the stream's 1,207,000 ft3/d baseline. Most of the extra water that
enters leaves through the stream: a higher sea raises the heads inland, and the
stream is where the aquifer discharges them.

Layers 4 and 5 hold 83.5 percent of the coastal conductance and carry 84.6
percent of the response, so a coastal boundary is only as good as the deep
conductance behind it. A 1 percent error in that conductance moves the coastal
flow by 522 ft3/d and the stream by 527 ft3/d, which is 0.04 percent of the
stream's baseline, so the boundary head is the term worth getting right.

A foot of sea-level rise is slightly more than a foot of boundary head here,
because the equivalent freshwater head rises 1.02 ft for every foot the sea does.

## Which parts of the aquifer control each outlet

The same backward solves also returned the sensitivity of each measure to
hydraulic conductivity in every cell. Map the composite `k11` sensitivity for the
stream and the coast in the middle layer, and correlate the two patterns.

In [ ]:
k11_sens = {
    measure: adjh.composite_sensitivity(ws, measure, "k11")
    for measure in ("swgw", "coastgw")
}
pair = [k11_sens[m][layer][active_cells] for m in ("swgw", "coastgw")]
print(f"correlation between the two patterns: {np.corrcoef(*pair)[0, 1]: .3f}")
for measure, values in zip(("swgw", "coastgw"), pair):
    ranked = np.sort(np.abs(values))[::-1]
    share = np.cumsum(ranked) / ranked.sum()
    print(
        f"{labels[measure]:8s} largest 10% of cells carry "
        f"{share[int(0.1 * ranked.size)]:.2f} of the total"
    )
for measure in ("swgw", "coastgw"):
    strongest = np.unravel_index(
        np.argmax(np.abs(k11_sens[measure][layer])), (nrow, ncol)
    )
    print(
        f"strongest cell for the {labels[measure]}: "
        f"x {xc[strongest]:,.0f} ft, y {yc[strongest]:,.0f} ft, worth "
        f"{k11_sens['swgw'][layer][strongest]:7.1f} to the stream and "
        f"{k11_sens['coastgw'][layer][strongest]:6.1f} to the coast"
    )

with flopy.plot.styles.USGSMap():
    fig, axd = plt.subplot_mosaic([["a", "b"]], figsize=(8, 6.5), layout="constrained")
    for measure, panel in (("swgw", "a"), ("coastgw", "b")):
        arr = k11_sens[measure][layer]
        vmax = np.percentile(np.abs(arr[active_cells]), 99)
        ax = axd[panel]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=layer)
        cb = mm.plot_array(arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        mm.plot_ibound()
        ax.plot(px, py, "k*", ms=14)
        ax.set_xlabel("x, in feet")
        flopy.plot.styles.heading(ax=ax, letter=panel, heading=labels[measure])
        fig.colorbar(
            cb, ax=ax, shrink=0.4, label="change in flow per ft/d, in ft$^3$/d"
        )
    axd["a"].set_ylabel("y, in feet")

**What to look for.** The two maps pick out much the same cells and give the
largest of them opposite signs, with a correlation of -0.625. The strongest cell
for the stream is on the stream itself, at x 4,250 ft and y 8,750 ft, worth -40.7
ft3/d per ft/d to the stream measure and 4.6 to the coast. The strongest for the
coast is the cell the stream leaves the valley through, at x 4,250 ft and y 250
ft, worth 23.8 to the coast and -24.2 to the stream.

The signs differ because the baselines do, not because the two outlets compete.
The stream takes water out of the aquifer and the sea puts it in, so conductivity
in the cells between them moves more water along the same path from the sea to
the stream, and both measures grow at once. The cells that control the coast are
the cells that control the stream.

For both measures, the tenth of the cells with the largest sensitivity carry 0.57
of the total, so the adjoint says where to spend the next field measurement, and
it names nearly the same cells for both.

## Check the adjoint with two model runs

The brute-force way to get a capture fraction is to run the model twice, once
with the well off and once with it on, and difference the budgets. That gives one
number per run, and it is an independent check on the adjoint.

Run the model again with the prediction well pumping at a small rate. Keep the
rate small: the adjoint returns a derivative evaluated at zero pumping, so the
difference only matches it in the limit of a small perturbation. Tighten the
solver at the same time, because the differences in lake flow are small enough to
be lost in ordinary convergence noise. Difference every budget term rather than
only the three measures, so the whole of the pumping is accounted for.

In [ ]:
dq = -3000.0  # ft^3/d, small enough to stay in the linear range

ws_base = adjh.prepare_model(
    "adj-coastal-q0",
    variant="coastal-advanced",
    prediction_rate=0.0,
    equivalent_freshwater=True,
    outer_dvclose=1.0e-10,
)
ws_pert = adjh.prepare_model(
    "adj-coastal-dq",
    variant="coastal-advanced",
    prediction_rate=dq,
    equivalent_freshwater=True,
    outer_dvclose=1.0e-10,
)
for w in (ws_base, ws_pert):
    adjh.run_model(w, mf6_exe)

print("outlet     adjoint   two-run   difference")
for measure, label in labels.items():
    diff = (
        adjh.budget_net(ws_pert, terms[measure])[last]
        - adjh.budget_net(ws_base, terms[measure])[last]
    ) / dq
    print(
        f"{label:8s} {capture[measure]:9.4f} {diff:9.4f} "
        f"{100 * abs(diff - capture[measure]) / abs(capture[measure]):9.1f}%"
    )

print("\nwhere every unit of pumping comes from, by budget term")
for term in ("SFR(SFR-1)", "GHB(GHB-1)", "UZF-GWRCH(UZF-1)", "LAK(LAK-1)"):
    share = (
        adjh.budget_net(ws_pert, term)[last] - adjh.budget_net(ws_base, term)[last]
    ) / dq
    print(f"  {term:18s} {share:8.4f}")
storage = (
    adjh.budget_net(ws_pert, "STO-SS(STORAGE)")[last]
    - adjh.budget_net(ws_base, "STO-SS(STORAGE)")[last]
) / dq
print(f"  {'STO-SS(STORAGE)':18s} {storage:8.4f}")

**What to look for.** The stream agrees with the two-run difference to 2.7
percent and the coast to 2.8 percent. Both agree to the same percentage at
perturbation rates of 3,000, 300, and 30 ft3/d, so the gap is not a
linearization error.

The budget decomposition says what the gap is. Every unit the well pumps is
supplied by the stream (0.3545), the sea (0.5820), UZF (0.0660), and aquifer
storage (0.0021), less the 0.0046 the lake no longer gives up, and those five add
to 1.0000. The UZF share is the one the adjoint cannot see: pumping lowers the
water table, so less of the infiltration is rejected as runoff and 0.0660 of the
pumping arrives as recharge that previously ran off the surface. mf6adj reported
at the start of the adjoint solve that it forms no terms for UZF or the mover, and
this is the consequence.

The same runoff is what gives the lake measure the wrong sign. The mover carries
UZF runoff to the lake, so less runoff lowers the lake stage and the lake leaks
less to the aquifer. A well near this lake takes less water from it than it did
before pumping, which the two-run difference puts at 0.0046 of the pumping and
the adjoint has as -0.0006. The lake terms are under 2 percent of the stream's
either way, so this well captures water from the sea and the stream whichever
number is used.

An adjoint sensitivity answers the question its formulation encodes, and mf6adj
reports which parts of that formulation are approximate. Where the difference
changes the answer, the two-run difference is the number to trust.

## Recap

- A **performance measure** names the model output you care about. Three
  measures, one per outlet, came out of a single forward run and three backward
  solves.
- A saltwater coast in a constant-density model is a boundary at the
  **equivalent freshwater head**, which rises from 0.061 ft in layer 1 to 5.8 ft
  at the bottom of the valley.
- The prediction well captures 0.566 of its pumping from the sea, 0.345 from the
  stream, and 0.0006 from the lake. Distance to the outlet decides the split, not
  the size of the outlet.
- Mapped over the grid, the same `wel6_q` sensitivity gives the capture fraction
  for a well placed anywhere. The sea supplies more than the stream on the
  seaward side of a divide 3,500 to 4,000 ft inland on the well's column, and
  over 20.1 percent of the valley.
- The sensitivity to the boundary head is the sea-level-rise question: a foot
  brings in 175,000 ft3/d more seawater and sends 149,000 ft3/d more to the
  stream, 84.6 percent of it through the two deep layers.
- Conductivity that makes the stream supply more of a well's water makes the sea
  supply less, with a correlation of -0.625 between the two sensitivity patterns.
- mf6adj warns when part of a package's derivative is not formed. Read those
  warnings, and check a measure with a small-perturbation two-run difference when
  they name something the model uses. Here UZF and the mover are what the stream
  and coast measures miss by under 3 percent and what gives the lake measure the
  wrong sign.